In [1]:
# 读取两个文件，去重并合并
import pandas as pd
#record1 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_07_04.xlsx", sheet_name=0)
record2 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_07_04.xlsx", sheet_name=1)
record3 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_07_04.xlsx", sheet_name=2)
record = pd.concat([record2, record3], axis=0)

#record = record[record['创建时间']>='2021-12-27']

e:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#record4 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_05_02.xls", sheet_name=0)
record5 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_05_02.xls", sheet_name=1)
record6 = pd.read_excel("H:\\bishe\\doc_info\\doctor_records_2022_05_02.xls", sheet_name=2)
record0 = pd.concat([record5,record6],axis = 0)

#record0 = record[record['创建时间']>='2021-12-27']

In [3]:
record = pd.concat([record,record0], ignore_index=True)
print(f"数据中重复的行数为: {record.duplicated().sum()}")
record = record.drop_duplicates()
record = record[record['创建时间']>='2022-01-03']
record = record[record['创建时间']<='2022-06-27']
print(len(record))

数据中重复的行数为: 1130874
160441


In [ ]:
#gift1 = record1[record1['时间']>='2022-01-03']
#gift1 = record1[record1['时间']<='2022-06-27']

#gift2 = record4[record4['时间']>='2022-01-03']
#gift2 = record4[record4['时间']<='2022-06-27']

#gift = pd.concat([gift1,gift2],ignore_index=True)
#gift = gift.drop_duplicates()

#record = record.merge(gift, on=['id', '用户id', '姓名'])
#record.to_csv('yuanshishuju.csv', index= False, encoding='utf-8-sig')

In [5]:
#定义周
def week_split(x):
    #if x>="2021-12-27" and x<"2022-01-03": return "01_03"
    if x>="2022-01-03" and x<"2022-01-10": return "01_03"
    elif x>="2022-01-10" and x<"2022-01-17": return "01_10"
    elif x>="2022-01-17" and x<"2022-01-24": return "01_17"
    elif x>="2022-01-24" and x<"2022-01-31": return "01_24"
    elif x>="2022-01-31" and x<"2022-02-07": return "01_31"
    elif x>="2022-02-07" and x<"2022-02-14": return "02_07"
    elif x>="2022-02-14" and x<"2022-02-21": return "02_14"
    elif x>="2022-02-21" and x<"2022-02-28": return "02_21"
    elif x>="2022-02-28" and x<"2022-03-07": return "02_28"
    elif x>="2022-03-07" and x<"2022-03-14": return "03_07"
    elif x>="2022-03-14" and x<"2022-03-21": return "03_14"
    elif x>="2022-03-21" and x<"2022-03-28": return "03_21"
    elif x>="2022-03-28" and x<"2022-04-04": return "03_28"
    elif x>="2022-04-04" and x<"2022-04-11": return "04_04"
    elif x>="2022-04-11" and x<"2022-04-18": return "04_11"
    elif x>="2022-04-18" and x<"2022-04-25": return "04_18"
    elif x>="2022-04-25" and x<"2022-05-02": return "04_25"
    elif x>="2022-05-02" and x<"2022-05-09": return "05_02"
    elif x>="2022-05-09" and x<"2022-05-16": return "05_09"
    elif x>="2022-05-16" and x<"2022-05-23": return "05_16"
    elif x>="2022-05-23" and x<"2022-05-30": return "05_23"
    elif x>="2022-05-30" and x<"2022-06-06": return "05_30"
    elif x>="2022-06-06" and x<"2022-06-13": return "06_06"
    elif x>="2022-06-13" and x<"2022-06-20": return "06_13"
    elif x>="2022-06-20" and x<"2022-06-27": return "06_20"
    #elif x>="2022-06-27" and x<"2022-07-04": return "07_04"

record['周'] = record['创建时间'].apply(lambda x: week_split(x))

In [6]:
record.to_csv('yuanshishuju.csv', index= False, encoding='utf-8-sig')

In [7]:
import re
#生成患者对话和医生对话
record['对话'] = record['对话'].fillna('')
record['医生对话'] = record['对话'].apply(lambda x: ' '.join(re.findall('\[医生\]: (.*?)\\n', x)))
record['患者对话'] = record['对话'].apply(lambda x: ' '.join(re.findall('\[患者\]: (.*?)\\n', x)))

## 颜值

In [11]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm  # 导入进度条库

# --- 1. 配置路径 ---
PROTO_PATH = r"H:\bishe\doc_info\deeep\models\resnext50_deploy.prototxt"
MODEL_PATH = r"H:\bishe\doc_info\deeep\models\resnext50.caffemodel"
IMAGE_DIR = r"H:\bishe\doc_info\pic" 

# --- 2. 加载模型 ---
print("正在加载 Caffe 模型...")
try:
    net = cv2.dnn.readNetFromCaffe(PROTO_PATH, MODEL_PATH)
    print("模型加载成功！")
except Exception as e:
    print(f"模型加载失败: {e}")

def cv_imread(file_path):
    """解决中文路径读取问题"""
    try:
        return cv2.imdecode(np.fromfile(file_path, dtype=np.uint8), -1)
    except Exception:
        return None

def get_caffe_score(img_name):
    img_name = str(img_name).strip()
    found_path = None
    for ext in ['.jpg', '.png', '.JPG', '.jpeg']:
        temp_path = os.path.join(IMAGE_DIR, f"{img_name}{ext}")
        if os.path.exists(temp_path):
            found_path = temp_path
            break
    
    if not found_path:
        return None
    
    try:
        image = cv_imread(found_path)
        if image is None: return None
        
        # --- 修复：处理 4 通道 (RGBA) 报错问题 ---
        if len(image.shape) == 2: # 灰度图
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        elif image.shape[2] == 4: # RGBA 图
            image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
        # ---------------------------------------

        blob = cv2.dnn.blobFromImage(image, 1.0, (224, 224), 
                                     (104, 117, 123), 
                                     swapRB=False, crop=False)
        net.setInput(blob)
        preds = net.forward()
        return round(float(preds[0][0]), 2)
    except Exception as e:
        # 如果还是报错，打印错误并跳过
        # print(f"跳过图片 {img_name}: {e}") 
        return None

# --- 3. 批量应用到 record (带进度条) ---

# 1. 获取唯一姓名并去除空值
unique_names = record['姓名'].dropna().unique()

# 2. 建立 姓名 -> 分数 的映射字典
name_to_score = {}

# 使用 tqdm 包装循环过程
# desc: 进度条前面的描述文字
# unit: 进度条的单位
print(f"开始处理，总计 {len(unique_names)} 个医生...")
for name in tqdm(unique_names, desc="颜值评分进度", unit="人"):
    score = get_caffe_score(name)
    name_to_score[name] = score

# 3. 将分数映射回 record 表
record['颜值'] = record['姓名'].map(name_to_score)

# --- 4. 结果统计 ---
success_count = record['颜值'].notna().sum()
print(f"\n处理完成！")
print(f"成功打分: {success_count} 条记录")
print(record[['姓名', '颜值']].head())

正在加载 Caffe 模型...
模型加载成功！
开始处理，总计 4547 个医生...


颜值评分进度: 100%|██████████| 4547/4547 [03:49<00:00, 19.80人/s]


处理完成！
成功打分: 160441 条记录
      姓名     颜值
20   冯光炜  38.95
194  魏彦照  32.51
198   李玲  34.08
249   李玲  34.08
250   李玲  34.08


## 文本

In [15]:
import re
record['对话'] = record['对话'].fillna('')
record['医生对话'] = record['对话'].apply(lambda x: ' '.join(re.findall('\[医生\]: (.*?)\\n', x)))
record['患者对话'] = record['对话'].apply(lambda x: ' '.join(re.findall('\[患者\]: (.*?)\\n', x)))

In [16]:
import jieba
import pandas as pd
import numpy as np  # 导入 numpy 处理空值
from tqdm import tqdm
import os

# 允许 tqdm 在 pandas 中使用
tqdm.pandas()

# --- 1. 初始化数据 ---
# 假设 record 已经定义好
data = record 

# --- 2. 加载词典 (逻辑保持不变) ---
medical_words_set = set()
try:
    with open("H:\\bishe\\refined_medical_dict.txt", encoding='utf-8') as f:
        for line in f.readlines():
            word = line.strip().split()[0]
            if len(word) > 1:
                medical_words_set.add(word)
    for w in medical_words_set:
        jieba.add_word(w)
    print(f"成功加载医学词典，共 {len(medical_words_set)} 个词")
except:
    print("⚠️ 未找到医学词典，医学密度计算可能不准！")

stop_words_set = set()
try:
    with open('H:\\bishe\\doc_info\\dict\\stopwords-master\\cn_stopwords.txt', encoding='utf-8') as f:
        for line in f.readlines():
            stop_words_set.add(line.strip())
except:
    stop_words_set = {'的', '了', '和', '是', '就', '都', '而', '及', '与'}
    print("⚠️ 未找到停用词表，使用内置基础停用词。")

# 词典定义 (保持不变)
certainty_words = {'肯定', '一定', '确诊', '绝对', '明显', '必须', '无疑', '明确', '百分之百', '确保', '显然', '根本', '立刻', '马上', '不用怀疑', '保证', '正是'}
uncertainty_words = {'可能', '大概', '也许', '好像', '恐怕', '估计', '不排除', '或许', '看情况', '观察', '疑似', '貌似', '大约', '可能吧', '不一定', '难说'}
politeness_words = {'您', '请', '祝', '麻烦', '谢谢', '感谢', '不用谢', '不客气', '没事', '没关系', '早日康复', '放心', '安心', '不用担心', '抱歉', '不好意思', '好的', '收到', '亲', '家长', '朋友'}

# ================= 3. 核心计算函数 (已修改) =================

def analyze_doctor_text_v2(text_content):
    """
    如果无法计算，统一返回 (None, None, None, None)
    """
    # 检查是否为字符串且不为空
    if not isinstance(text_content, str) or len(text_content.strip()) == 0:
        return None, None, None, None

    try:
        # 分词
        words = jieba.lcut(text_content)
        
        total_valid_words = 0
        medical_cnt = 0
        certainty_cnt = 0
        uncertainty_cnt = 0
        politeness_cnt = 0
        
        for w in words:
            # 跳过停用词和标点
            if w in stop_words_set or len(w.strip()) == 0:
                continue
            
            total_valid_words += 1
            
            if w in medical_words_set:
                medical_cnt += 1
            if w in certainty_words:
                certainty_cnt += 1
            elif w in uncertainty_words:
                uncertainty_cnt += 1
            if w in politeness_words:
                politeness_cnt += 1

        # 如果没有有效词，无法计算比例，返回 None
        if total_valid_words == 0:
            return None, None, None, None
        
        # 计算指标
        medical_density = medical_cnt / total_valid_words
        certainty_score = (certainty_cnt - uncertainty_cnt) / total_valid_words
        politeness_density = politeness_cnt / total_valid_words

        return medical_density, certainty_score, politeness_density, total_valid_words

    except Exception as e:
        # 捕获任何意外错误并返回 None
        return None, None, None, None

# ================= 4. 执行计算并保存 =================

print("正在计算文本指标...")

# 使用 progress_apply 替代 apply 以显示进度条
results = data['医生对话'].progress_apply(lambda x: pd.Series(analyze_doctor_text_v2(x)))

# 重命名列名
results.columns = ['medical_density', 'certainty_score', 'politeness_density', 'word_count']

# 将结果合并回原表
data = pd.concat([data, results], axis=1)


成功加载医学词典，共 13782 个词
正在计算文本指标...


100%|██████████| 160441/160441 [01:49<00:00, 1468.53it/s]


In [ ]:
data.to_csv('yanshengbianliang.csv', index= False, encoding='utf-8-sig')

In [6]:
record = pd.read_csv('yanshengbianliang.csv')

## 声音

In [ ]:
import pandas as pd
import re
import io
import requests
import librosa
import torch
import torch.nn.functional as F
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import tempfile  # 必须导入
import os        # 必须导入
import warnings
warnings.filterwarnings("ignore")

# 1. 环境与模型初始化
model_name = "/mnt/biye/model/" 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在加载模型至 {device}...")

processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

ID2LABEL = model.config.id2label
print(f"模型标签映射: {ID2LABEL}")

def download_and_preprocess(url):
    """通过临时文件解决 MP3 解码失败问题"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    tmp_path = None
    try:
        # 1. 下载音频
        response = requests.get(url, timeout=15, headers=headers)
        if response.status_code != 200:
            return None
        
        # 2. 创建临时文件
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_file:
            tmp_file.write(response.content)
            tmp_path = tmp_file.name
        
        # 3. 加载音频
        y, sr = librosa.load(tmp_path, sr=16000)
        
        # 4. 截断音频（保留前 10 秒）
        if len(y) > 16000 * 10:
            y = y[:16000 * 10]
        return y

    except Exception:
        return None
    finally:
        # 5. 清理临时文件
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except:
                pass

def get_emotion_batch(y_list):
    results = []
    for y in y_list:
        if y is None or len(y) < 1600:
            results.append(None)
            continue
        inputs = processor(y, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
        try:
            with torch.no_grad():
                logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)[0].cpu().tolist()
            scores = {ID2LABEL[i]: p for i, p in enumerate(probs)}
            
            p_happy = scores.get('hap', 0)
            p_angry = scores.get('ang', 0)
            p_sad = scores.get('sad', 0)
            p_neutral = scores.get('neu', 0)
            
            v = p_happy - (p_angry + p_sad)/2
            a = (p_happy + p_angry)/2 - p_neutral
            results.append({'v': v, 'a': a, 'hap': p_happy, 'ang': p_angry, 'sad': p_sad, 'neu': p_neutral})
        except: 
            results.append(None)
    return results

def process_row_optimized(text, executor):
    if not isinstance(text, str): return None
    
    # 宽松正则
    pattern = r'\[医生\][:：]?\s*\[语音\((https?://[^\)]+)\)\]'
    urls = re.findall(pattern, text)
    if not urls: return None
    
    y_audios = list(executor.map(download_and_preprocess, urls))
    valid_audios = [y for y in y_audios if y is not None]
    if not valid_audios: return None
    
    emotion_res = get_emotion_batch(valid_audios)
    valid_res = [r for r in emotion_res if r is not None]
    if not valid_res: return None
    
    avg_metrics = {}
    for key in ['v', 'a', 'hap', 'ang', 'sad', 'neu']:
        avg_metrics[key] = sum(r[key] for r in valid_res) / len(valid_res)
    return avg_metrics

# --- 执行流 ---

record = pd.read_csv('yanshengbianliang.csv')

# 初始化新列为 0.0
for col in ['valence', 'arousal', 'prob_happy', 'prob_angry', 'prob_sad', 'prob_neutral']:
    record[col] = 0.0

print(f"\n开始正式处理，总行数: {len(record)}")
success_count = 0

# 开启多线程处理
with ThreadPoolExecutor(max_workers=4) as executor:
    for index, row in tqdm(record.iterrows(), total=len(record), desc="深度情感分析中"):
        res = process_row_optimized(row['对话'], executor)
        
        if isinstance(res, dict):
            record.at[index, 'valence'] = res['v']
            record.at[index, 'arousal'] = res['a']
            record.at[index, 'prob_happy'] = res['hap']
            record.at[index, 'prob_angry'] = res['ang']
            record.at[index, 'prob_sad'] = res['sad']
            record.at[index, 'prob_neutral'] = res['neu']
            success_count += 1

# 保存结果
record.to_csv("ysbl_new.csv", index=False, encoding='utf-8-sig')
print(f"\n处理完成！共成功识别并填充了 {success_count} 行数据。")
print("结果已保存至 ysbl_new.csv")

## 将最后文件格式调整

In [11]:
import pandas as pd
yuanshi = pd.read_csv('yuanshishuju.csv', engine='python', encoding='utf-8-sig')
yuanshi['创建时间'] = pd.to_datetime(yuanshi['创建时间'])
yuanshi_new = yuanshi.sort_values(by=['id', '创建时间'], ascending=[True, True])
yuanshi_new = yuanshi_new.reset_index(drop=True)
yuanshi_new.to_excel('yuanshi_new.xlsx', index=False, engine='openpyxl')

In [12]:
ysbl = pd.read_csv('ysbl_new.csv', engine='python', encoding='utf-8-sig')
ysbl['创建时间'] = pd.to_datetime(ysbl['创建时间'], errors='coerce')
ysbl = ysbl.dropna(subset=['创建时间'])
ysbl_new = ysbl.sort_values(by=['id', '创建时间'], ascending=[True, True])
ysbl_new = ysbl_new.reset_index(drop=True)
ysbl_new.to_excel('ysbl_new_new.xlsx', index=False, engine='openpyxl')

In [6]:
import pandas as pd
df = pd.read_excel("ysbl_new_new.xlsx", engine="openpyxl")

# 2. 删除无用的患者「用户id」列（严格按你的要求）
df = df.drop(columns=["用户id"])

# 3. 定义分组键：【医生id + 周】（核心修正！）
group_cols = ["id", "周"]

# 4. 自动定位起始列：从medical_density（截图里是ical_denstainty_s）开始到最后一列
# 这一列之后的所有变量都按周求平均
start_col = "medical_density"
start_idx = df.columns.get_loc(start_col)
avg_columns = df.columns[start_idx:]  # 需平均的所有列

# 5. 分组计算：id+周分组，姓名取唯一值，数值列求平均
weekly_avg = df.groupby(group_cols, as_index=False).agg(
    {"姓名": "first"}  # 每个医生id对应唯一姓名
    | {col: "mean" for col in avg_columns}  # 后续列全部周平均
)

# 6. 按【医生id降序】排序（你的要求）
weekly_avg = weekly_avg.sort_values(by="id", ascending=False)

# 7. 调整列顺序：id → 姓名 → 周 → 平均变量（整洁规范）
final_cols = ["id", "姓名", "周"] + avg_columns.tolist()
weekly_avg = weekly_avg[final_cols]

# 8. 保存最终结果
weekly_avg.to_excel("医生每周统计结果.xlsx", index=False, engine="openpyxl")

# 打印预览（验证结果）
print("✅ 统计完成！最终结果预览：")
print(weekly_avg.head(10))

✅ 统计完成！最终结果预览：
                         id   姓名      周  medical_density  certainty_score  \
43472  ffed60310f01c1caedbf  周文颖  04_04              NaN              NaN   
43464  ffd256862fb069b4ed10  王冬梅  02_28         0.094840         0.000000   
43457  ffd256862fb069b4ed10  王冬梅  01_03         0.108730         0.005242   
43458  ffd256862fb069b4ed10  王冬梅  01_17         0.159243         0.006944   
43460  ffd256862fb069b4ed10  王冬梅  01_31         0.097096        -0.012560   
43461  ffd256862fb069b4ed10  王冬梅  02_07         0.072198         0.005359   
43462  ffd256862fb069b4ed10  王冬梅  02_14         0.061224         0.000000   
43463  ffd256862fb069b4ed10  王冬梅  02_21         0.115149        -0.003175   
43459  ffd256862fb069b4ed10  王冬梅  01_24         0.108333         0.006667   
43465  ffd256862fb069b4ed10  王冬梅  03_14         0.096774        -0.016129   

       politeness_density  word_count   valence   arousal  prob_happy  \
43472                 NaN         NaN  0.000000  0.000000    0.0